## CASH Notebook

## Celestial Chase - LVL 1: The Teal Beacon

You've just woken up from cryo-sleep. No memory. No crew. Just you, a dying sun, and a faint signal pulsing from the sky.

One star glows different from the rest. Not white. Not grey. **Teal.**

Find it. Its position holds part of the key. But position alone is not enough - you must also know how much of its face is lit by the sun.

---

**The encoded signal:** `pool`

**Your task:**
1. Display the image and find the **cyan pixel** - it is the only pixel where `B == 255` and `G == 255` and `R == 0`
2. Read its `(x, y)` coordinates
3. Use `ephem` to compute **Venus's phase** (`int(venus.phase)`) on `2025/6/21 12:00:00` UTC from Zurich (lat=`47.3769`, lon=`8.5417`)
4. Build the key: `key = x * y + int(venus.phase)`
5. Build the permutation: `random.Random(key).shuffle(list(range(len(encoded))))`
6. Reverse the transposition to decode: `decoded[i] = encoded[perm[i]]`


In [ ]:
import base64, cv2, numpy as np
from IPython.display import display, Image as IPImage

# Starfield image embedded directly in this notebook
img_b64 = "iVBORw0KGgoAAAANSUhEUgAAAZAAAAGQCAIAAAAP3aGbAAAVUklEQVR4Ae3BCXDX9Z3/8dd75q+72q52xQO3YEWLghcoolZauZFTziQQIEASjoQzQDjDHc4A4UwIJAECBJJwyik37WJVRAEPUKpYoSseuNVtddf+Z/jP+B93dDD4C/kd38/v+3w8TADgCBPgMd2HDF63aLGAK5gAwBEmAAieo2dON6hVW6FhAgBHmADAESbPOPflFzVuulnlS52QkTMtU4BTug0auH7JUiEYTN5w8OSJJnXq6loNnDhh6dRpAhDVTADgCBPc17BTxyObt6gcO/74YtvfPC3AfSYAcIQJABxhAgBHmADAESYAUW3t7l09WrVWVDABgCNMACAd/+BcvbtryNtMQJQ6dOpk40fqCFHEBACOMCESJizInjYsTYi0l86++1TN+wRHmAAgLLIKC9ITk1QJJgDwtozs+ZlpwyWZAMARJgBwhOlaNezU8cjmLQKAcLFlZaUDYmIFuO/Nj/7joTv/TYheJmfte+1488fqCYBvmACE0trdu3q0ai0EgwkAHGECoku3QQPXL1kqRCMTADjCBIRY09iYA6VlAirNBACOMAHRomDrlqQOHYXoZfKBRp07Hd60WQDKt3zTxn6du8jbTADgCBMQSrmlJSmxcQKCwQQAjjABgCNMAKJL4batie07KBqZfkpcakpJTq6+p31in22FKwUA4WWKIqc/+bj27XcIQJQyAYAjTM76y39//ct/vkGAv53+5OPat9+hIJm0aOGUIUPlVaYfM27e3BkjRgrwjLOfX6p5SxXB30yABxw48XrTuo8KuCoTADjCBKB8iekjC7PmCkHy0tl3n6p5n66VCQAcYQIAR5hQCfGDBxUvXiIgGqVkjM/NnC4vMQGA563esb1X23YmfCshbVhR9gIBFdEkpsvBso1CuJgAwBEmuKxo546ENm0VSqUH9sc2bSaf6Tls6JoFCwWPMcF/uvTvtzFvuSKtaOeOhDZtBQTMBATJ9hePtnu6gYCQMQGAI0wA4AgTADjCBAARUnpgf2zTZgqYCQAcYQIAR5gAb0sePSp/9hwBkinYzn5+qeYtVQQAwWYCAEeY4LJWPbrvXrtO7tjy+yMdn2mo8Fq7e1ePVq0F95m+U3pgf2zTZkKw7X7l5VZPPCkAldAhKXFrQaEJ+J5bLl/+3ExAkBS/sCf+2ZYKEhMAOMIEOCIje35m2nDBx0xw2aJ1a4d07yHABe989un9t96mSjAhKrRJ6LmzaI2AqGYC4Igu/fttzFuuEJu0aOGUIUPlSSYAcISpfCcvnK9TrboA+NiMvGXj+g+QN5gAwBEmACjH2t27erRqLc8wRcL0ZbnjB6QIACrCBEc0ielysGyjAB8zAfCNc19+UeOmm/WdpRvWD+zaTe4wAUCwde7Xd9PyFQo2E1Bx81atHNG7j4DwMjmuY3LSlvwCVVzT2JgDpWUC/KT0wP7Yps3kLBP8LSVjfG7mdEWv9ol9thWuFCooefSo/Nlz5DEmAHCEqXzvXvrsviq3CgCuasO+vV2bt1DomfBDo+fMnj1qtAB4jwkAHGECEEZxqSklObnCNTEB3tBj6JC1CxcJKJ8JQFRLnZCRMy1ToTc7f8Xo5L4KJRMQgOMfnKt3dw0BEWUCHFSwdUtSh45CMOw9/mqLeo/LBSYAcIQJABxhKsfEhQumDh0mAPAMEwA4wvRDF//xTdXrrhcAeI8Jntdj6JC1CxcJUapFt65712+Q9yxZXzyoW7y8xAT80MZDB7s0biLAe0wAEAxlBw/ENGmqUDIFYMn64kHd4gUAEWXypSqXL18yE67Vha+/qnbDjQLCywQAjjABvrTn2Cst6z8hOMUEAI4wRa/BkyctnjxFwDU5+/mlmrdUEbzEhLCYkbdsXP8BAlAJJkSvo2dON6hVW+Gy6fChzo0aC46Yu7JwZJ9EOcUERNr81auG9+ot4AqZuTkZKan6jsmTug8ZvG7RYoXR3JWFI/skCq5JSBtWlL1A8AcTvOqtix89WPVOAfiOKRhGzJg+b9x4Af7wbHy3F4rXC2FnAuAN+Vs2J3fsJJTP5A2DJ09aPHmKAKB8JnzPy386++SvawqAJ5mAqBAzoH/ZsjwhqpkAeNVzfXo/v3KVfKBNQs+dRWv0U0wA4AgTAHjezOV5Y/v1NwHue/3DPz9616+EaGcCAEeYAFTe5csyE0LM5Ht5G8v6d4kRELDc0pKU2Dj9r8uX9f+ZCaFkQpAkpo8szJorfM/RM6cb1KotP7h8WWZCiJkQetNylk5IHahQmlOQPyopWXDNzOV5Y/v1FwJj8r0jb77R8KGHBR/oOjB1w9IcVU520eq0hF5CKHVIStxaUKgrmADAESYAcIQJgPecvHC+TrXqCqURM6bPGzdeTjF53tSlSyYOHCTgW8s3bezXuYsQXmmZ07IzJijSTNElLXNadsYEAWF07P336t9zrxB6JgBwhAnwpeMfnKt3dw3hqlY+v63Pc+3lGaYfk1VYkJ6YJN9Ys2tnz9ZtBMDbTPC9jYcOdmncRIFZsXlT306dBUSCCXDQoEkTl0yZKviMCQAcYQIAR5gQMsfef6/+PfcKcN+waVMXTJioSDMh0kbMmD5v3HjB3/Yce6Vl/SeEqzIBgCNMAOAIEwA4wgREqaUb1g/s2k3Bs3zTxn6duwiRY0IwnPn0k1q33S4AoWQCPGbs3KyZI9MFXMEEAI4wVUR20eq0hF5CGL118aMHq96pYDhw4vWmdR8V4CyT5zXs1PHI5i0CIurDv//trp/9XIgokwec+suFR35ZTVd47c8fPParuwUA3zJFowtff1XthhsFILqYAMARJgAe0HVg6oalOcJVmeAncakpJTm5gj8kpo8szJqrYNh85HCnho0UmGZxsftLShUCpiC58PVX1W64UfCHQ6dONn6kjjwvt7QkJTZOiBam0GjZPX7PumIFW5f+/TbmLRcAXzL5w4Wvv6p2w40C4D1pmdOyMyYoACYAcIQJABxhAgBHmADAESYfW1y8bnB8dwFwhAkAHGHyjU59kzevyBcAZ5kAwBGmsCvauSOhTVsBAVi6Yf3Art2Ea9U0NuZAaZkq59RfLjzyy2ryABP8JCVjfG7mdAFuMgGAI0yAh539/FLNW6oI/rD9xaPtnm6gcvQanmYCosu7lz67r8qtQjQyeUbexrL+XWIEAOUwAYAjTIE59+UXNW66WUAwjJ8/b/rwEQIqyATAY3a9/FLrJ58SrmAC/GT0nNmzR40W3GQCAEeYEIBBkyYumTJVACLKBACOMCEwh06dbPxIHUXaf/zPf//bP/2z/KRNQs+dRWsESCYAAfvom/+58/p/UkS989mn9996m3zJBHjGnIL8UUnJgr/Nzl8xOrmvfowJCItNhw91btRYQCWYwmXPsVda1n9CFbd6x/ZebdsJFffBf31597/cJHjb2t27erRqLQTABACOMCESinbuSGjTVogiZz+/VPOWKkIomQDAESa4pmNy0pb8AgH+Y0Ll7HvtePPH6glA6JngVS+dffepmvfJlzYfOdypYSMBP2QCAEeYAHjeml07e7ZuI98zIXLaJ/bZVrhScNa8VStH9O4jhIsJADzsuT69n1+5St8yAcG2dveuHq1aCwg2EwAEYNX253u3e04RZQJcMHjypMWTpwh+8vu33nzmwYf0PSYAcITJEX3HjF4xa7Z8oPeI4avmzReAK5gAwBEmBNvmI4c7NWwkj7n4j2+qXne9AJeZgCsUv7An/tmWQuU06tzp8KbNQvCYAMARJsARL75z5un7a8nDRs6cMXfsOCFkTADgYdlFq9MSeulbJgBwhMmrWnTrunf9BgHAd0z+s+nwoc6NGgv+kJg+sjBrrhAVTADgCBPglE59kzevyBd8yRTt1uza2bN1GwFwnwmIXh2SErcWFArRwuRtM/KWjes/QID7+o0ds3zmLH2n+IU98c+2FCrCBMB7cktLUmLjFEU69U3evCJflWMqX9PYmAOlZUKItevda/uq1QJCbMcfX2z7m6flMhMCs6ysdEBMrABEjgkAvrP/9deaPfqYvMoEAI4wIXg27NvbtXkLed7hN041evgRAa4xAYAjTChf5359Ny1fIaB8Y7LmzEofJYSFKRrFDOhftixPQJR68Z0zT99fS/5jCoaS/fvimjUXvGHnS39s89RvBLjg4j++qXrd9QqMyfcOnjzRpE5dAfA8EwA4wgQEIGlUesGcLAERZQIAR5gARLu9x19tUe9xhV2v4Wmr52creExwxPtf/PWem38hwMdMgLNiBvQvW5Yn+IYJABxhAhAka3bt7Nm6jRAyJgBwhAkAPGP7i0fbPd1A5TAh0va9drz5Y/Xkmj+8/dbvHnhQKN+YrDmz0kcpAJm5ORkpqcJPMYVey+7xe9YVy8PW7dndvWUrRZ1mcbH7S0oVRks3rB/YtZuA0DABgCNMAJy1//XXmj36mHzDhIhqFhe7v6RU8JOhU6csnDhJvpE+a2bWmLEKBhPCrmDrlqQOHQUE5vgH5+rdXUOe8af//PzX/3qLIsFUCW9d/OjBqncKAMLCBACOMME1BVu3JHXoKK9asr54ULd4Addk7/FXW9R7XOUwISw27NvbtXkLecmJ8x/WrX6X4Kzh0zPnj8+Qn5jCZe/xV1vUe1wAQqDX8LTV87MV7Uzwh38//fZvaz8gVNpLZ999quZ9QiSY4GNzVxaO7JMowBEmwMMS0oYVZS8QQq9tr4Qdq4vkbSYAcIQJgDu69O+3MW+5/MoEAI4wVcS/n377t7UfkMve+ezT+2+9TUBEvffX/7z3F/+qoCrZvy+uWXNFNRMAOMLkAQPGj1s2fYYA4KpMADwvb2NZ/y4xulbH3n+v/j33yn0mAHCECYALkkePyp89R/5mAkJv9ysvt3riSQGVYwJCLyN7fmbacCEY3v/ir/fc/AuF3awVy8f07aeIMgGAI0yR033I4HWLFgsAAmMCAEeYIuqtix89WPVOAUAATIAP5JaWpMTGCeF14MTrTes+quAxuSZ+8KDixUsEwH9MoXfT5ctfmgkAKsfkJdOX5Y4fkCIA+DEmIBqd/+rv1W/8mX7KgPHjlk2fITjC5G8jZ86YO3acALjABKAS/vy3//rVz/9FCAsTADjCBPzQgROvN637qADvMQXDhn17uzZvIQAh89LZd5+qeZ/8zQQAjjAB0WLQpIlLpkwVopfJKaPnzJ49arSAinj5T2ef/HVNwX0mAHCECS77w9tvXbp0qcPvnhHgbXGpKSU5uaocEwA4woRrsrh43eD47gIQRiYAcIQJHtOqR/fda9cJwBVMAOAIEwA4wlQRB0+eaFKnrgBEQlZhQXpi0pL1xYO6xcuXTACCZO3uXT1atVZUW7CmaFjPBAXg6JnTDWrVVlCZos5zfXo/v3KVgMC8/fHFB+6oKrjA5FX9x43NmzFTfpI8elT+7DkCwi6rsCA9MUmeZwIAR5gAwBEmAE554dVjzz5eX75kQqUtWV88qFu8AtZ1YOqGpTkCUEEmAL43dOqUhRMnqSJOnP+wbvW7FF4mIOrklGxIjeuqH3Ps/ffq33Ov4CYTAF/a8vsjHZ9pKKeYAMARJgCRcPbzSzVvqSJUhAkIi3a9e21ftVrANTny5hsNH3rYBACOMAH+dsPly1+bCS4wAUAwjJ2bNXNkukLJBACOMLns+Afn6t1dQwD8wQREqe5DBq9btFiIIiYAcIQJETJ6zuzZo0YLQMBMACKn+5DB6xYtFgJjAqLF8OmZ88dnCNHLhOg1demSiQMHCQi2tbt39WjVWmFnAuC+5l3j9m0oUbQzIaIysudnpg0XgACYvCe3tCQlNk74oVkrlo/p20+Aj5kAwBEmAB5TtHNHQpu2whVMQMgUbtua2L6DEFETFmRPG5amqGBCeDXvGrdvQ4kAVJwJABxhwo+58PVX1W64UQC8xAR3HH7jVKOHHxHgVybAWYXbtia27yB8z8ZDB7s0bqIoZQIAR5gAwBEmAHCECQAcYYK0rKx0QEysgApqn9hnW+FKIVxModRv7JjlM2cJAILBhEo4euZ0g1q1da2mL8sdPyBFAAJjAgBHmBBeKRnjczOnC0DFmQC4rGGnjkc2b5Hj2vXutX3Vav0UEwA4wgQAjjABkbBo3doh3XsIIXPywvk61aorupgAeNXUpUsmDhwkSEOmTF40abIpBLIKC9ITkwQAQWUCgBDrmJy0Jb9AlWZCYDr367tp+QoBiByTm155709P3PtrAZGQkT0/M224giGnZENqXFchMCYAXrJm186erdsIP8YEAI4whd6RN99o+NDDAqLaoVMnr7vuut/WfkBR59yXX9S46WZ5gAkAHGGKqLc/vvjAHVUFAAEwAYAjTI47/cnHtW+/Qwi2o2dON6hVW3Bfh6TErQWFulaZuTkZKanyBhMAOMIEAI4wwTdWbN7Ut1NnAR5QtHNHQpu2qiATQqNpbMyB0jIBCB4TUAk9hw1ds2ChgLAwAVGhaOeOhDZtVTl7jr3Ssv4TgleZAMARJgBwhAkAvjV06pSFEyfJw0xXlV20Oi2hl3BNTl44X6dadQEIEhNcNjt/xejkvgL8wRQWH/79b3f97OcCEEUu/uObqtddrzAyAYAjTADgCFNQLS5eNzi+uwAgBEyVcPiNU40efkSBOfPpJ7Vuu10AcK1McErT2JgDpWUC3LH1D7/v8LtnFAwmRJ2pS5dMHDhIQNQxAYAjTAAcN3Zu1syR6aqEsXOzZo5Ml+eZAMARplCKGdC/bFmeECQL164Z2qOnAL8yAfC9wm1bE9t3kOeZgEoYN2/ujBEjBYSFKQAdk5O25BcIACLKhCv0Gztm+cxZqoS3P774wB1V9VNado/fs65YAAJjAgBHmPxqWs7SCakDBcAdJgARkl20Oi2hl8IlZkD/smV5+qG41JSSnFw5wgQAjjABgCNMAHBVF77+qtoNN8oDTADCa8cfX2z7m6d1VemzZmaNGSv8kAkIhoETJyydOk1AKJkAwBEmVERWYUF6YpIqaOTMGXPHjhOuSfLoUfmz5wiQTADgCBMAOMLkJyX798U1ay4gjHa9/FLrJ58SgsEEBFvxC3vin20pIHgS00cWZs01AYAjTADgCBO+5+SF83WqVRcATzL5z+Yjhzs1bCQE2+lPPq59+x0CQsYEwJOSRqUXzMkSvseE8sWlppTk5AqR838uX/6/ZgK+ZQKC5K2LHz1Y9U4BIWOKRo06dzq8abMARBcTAITRuj27u7dspWti8pkhUyYvmjRZABxkAgBHmADAESYXjMmaMyt9lAB4yYnzH9atfpcqIS41pSQnVwEzAYAj/h+baVUezADYmAAAAABJRU5ErkJggg=="

img_bytes = base64.b64decode(img_b64)
img_arr   = np.frombuffer(img_bytes, dtype=np.uint8)
img       = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)

cv2.imwrite('starfield.png', img)
display(IPImage('starfield.png'))
print("Image shape:", img.shape)


In [ ]:
import ephem, random

encoded  = "pool"
obs_date = "2025/6/21 12:00:00"
obs_lat  = "47.3769"
obs_lon  = "8.5417"

# TODO Step 1: Find the cyan pixel (B=255, G=255, R=0) in img
# Hint: use np.where or a loop
pixel_x, pixel_y = 0, 0  # replace with actual coordinates

# TODO Step 2: Compute Venus phase with ephem
obs = ephem.Observer()
obs.lat  = obs_lat
obs.lon  = obs_lon
obs.date = obs_date
venus = ephem.Venus()
venus.compute(obs)
phase = 0  # replace: int(venus.phase)

# TODO Step 3: Build key and permutation
key  = pixel_x * pixel_y + phase
perm = list(range(len(encoded)))
random.Random(key).shuffle(perm)

# TODO Step 4: Reverse the transposition
# Hint: if encoded[perm[i]] = original[i], then decoded[i] = encoded[perm[i]]? Think carefully.
def transpose_decode(encoded, perm):
    pass  # implement this

answer = transpose_decode(encoded, perm)
print(answer)
